# Calculation of Carbon Contract for Difference (CCfD)

This notebook calculates the carbon price of several fossil fuel alternatives.
- Methanol
- Methane
- Ammonia
- Hydrogen
- Diesel
- Gasoline
- Jet_Fuel

## Setup

In [72]:
%run -i functions.py

## Load variables

In [75]:
# Load constant variables
constants = pd.read_csv('constants.csv', delimiter = ';', header = None, names=['variable', 'value'], index_col='variable')['value'].to_dict()
energy_density = constants['energy_density']

# Load Emission Factors (EF) of electrcity from the grid and the fossil fuel alternatives
df_fossil_ef, df_grid_ef = load_emission_factors("test_data/emission_factors_fossil_grid.xlsx")

# Load market and EUA (EU Allowence) prices of fossil fuel alternatives
df_fossil_prices, fossil_prices_currency, df_eua = load_cost_factors("test_data/cost_factors_fossil_alternatives.xlsx")

# Load exchange rates
df_exchange_rates = load_exchange_rates("test_data/exchange_rates.xlsx")

# Load results
results = load_results("test_data/results.xlsx")

### Check import of variables

In [4]:
energy_density

19700000000.0

In [66]:
df_fossil_ef

,Upper range,Lower range
Methanol,2.863,2.05
Methane,2.863,2.05
Ammonia,2.863,2.05
Hydrogen,2.863,2.05
Diesel,2.863,2.05
Gasoline,2.863,2.05
Jet_Fuel,2.863,2.05


In [6]:
df_grid_ef

,DK1,DK2,DK_BHM,FI,NO_1,NO_2,NO_3,NO_4,NO_5,SE_1,SE_2,SE_3,SE_4,DE
2018,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2019,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2020,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2021,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2022,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2023,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2024,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2025,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8,499.8
2026,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0
2027,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0,450.0


In [7]:
df_fossil_prices

,Methanol,Methane,Ammonia,Hydrogen,Diesel,Gasoline,Jet_Fuel
Cost,,,,,,,
2018,370.0,370.0,370.0,370.0,370.0,370.0,370.0
2019,205.0,205.0,205.0,205.0,205.0,205.0,205.0
2020,395.0,395.0,395.0,395.0,395.0,395.0,395.0
2021,399.0,399.0,399.0,399.0,399.0,399.0,399.0
2022,321.5,321.5,321.5,321.5,321.5,321.5,321.5
2023,370.0,370.0,370.0,370.0,370.0,370.0,370.0
2024,370.0,370.0,370.0,370.0,370.0,370.0,370.0
2025,370.0,370.0,370.0,370.0,370.0,370.0,370.0
2026,370.0,370.0,370.0,370.0,370.0,370.0,370.0


In [8]:
df_eua

{2018: 24.723125,
 2019: 24.38660287,
 2020: 24.38660287,
 2021: 54.15156951,
 2022: 80.18404545,
 2023: 83.59650224,
 2024: 83.59650224,
 2025: 83.59650224,
 2026: 83.59650224,
 2027: 83.59650224,
 2028: 83.59650224,
 2029: 83.59650224,
 2030: 83.59650224,
 2031: 83.59650224,
 2032: 83.59650224,
 2033: 83.59650224,
 2034: 83.59650224,
 2035: 133.6662,
 2036: 133.6662,
 2037: 133.6662,
 2038: 133.6662,
 2039: 133.6662,
 2040: 133.6662,
 2041: 133.6662,
 2042: 133.6662,
 2043: 133.6662,
 2044: 133.6662,
 2045: 133.6662,
 2046: 275.7618,
 2047: 275.7618,
 2048: 275.7618,
 2049: 275.7618,
 2050: 275.7618}

In [3]:
results

,hub_costs,electricity_from_grid,res_from_hub,pv_from_ppa,wind_from_ppa,revenue_side_products,revenue_res,demand
2019,1415.70,{'DK1': 115347.54452986868},131805.9000,0,0,"{'oxygen': 0, 'distrcit_heating': 0}",0,32000
2020,1205.35,{'DK1': 119417.97961992264},128637.7488,0,0,"{'oxygen': 0, 'distrcit_heating': 0}",0,32000


## Run the analysis 

In [76]:
fossil_fuel_alternative = "Methanol"
year = 2019

demand = results.loc[year, 'demand_t']
elec_breakdown = compute_electricity_breakdown(
    results.loc[year, 'pv_from_hub'],
    results.loc[year, 'wind_onshore_from_hub'],
    results.loc[year, 'wind_offshore_from_hub'],
    results.loc[year, 'pv_from_ppa'],
    results.loc[year, 'wind_from_ppa'],
    results.loc[year, 'electricity_from_grid'],
)

total_emissions_tCO2, emissions_per_sourcer_tCO2 = compute_hub_emissions(
    year,
    results.loc[year, 'pv_from_ppa'],
    results.loc[year, 'wind_from_ppa'],
    results.loc[year, 'electricity_from_grid'],
    df_grid_ef,
    0)

emissions_indstr_low_t, emissions_indstr_high_t  = compute_fossil_emissions(
    fossil_fuel_alternative,
    demand,
    df_fossil_ef
)

costs_without_revenues, _, _, total_revenues, costs_net_after_revenues = compute_hub_costs(
    results.loc[year, 'hub_costs'],
    results.loc[year, 'revenue_side_products'],
    results.loc[year, 'revenue_res'],
    demand
)

ff_alternative_costs = compute_industrial_costs(
    year,
    fossil_fuel_alternative,
    demand,
    df_fossil_prices,
    df_exchange_rates,
    fossil_prices_currency
)

CP_low_no_revs = compute_carbon_price(costs_without_revenues, ff_alternative_costs, total_emissions_tCO2, emissions_indstr_low_t)
CP_high_no_revs = compute_carbon_price(costs_without_revenues, ff_alternative_costs, total_emissions_tCO2, emissions_indstr_high_t)
CP_low_with_revs = compute_carbon_price(costs_net_after_revenues, ff_alternative_costs, total_emissions_tCO2, emissions_indstr_low_t)
CP_high_with_revs = compute_carbon_price(costs_net_after_revenues, ff_alternative_costs, total_emissions_tCO2, emissions_indstr_high_t)

row = {
    'scenario': 'Base',
    'year': year,
    'demand_t': demand,
    'total_elec_MWh': elec_breakdown['total_elec_mwh'],
    'res_mwh': elec_breakdown['res_used_mwh'],
    'ppa_mwh': elec_breakdown['ppa_total_mwh'],
    'grid_mwh': elec_breakdown['grid_total_mwh'],
    'supply_mismatch_MWh': 0,
    'total_emissions_tCO2': total_emissions_tCO2,
    'emissions_tCO2_per_t': total_emissions_tCO2 / demand,
    'emissions_indstr_low_t': emissions_indstr_low_t,
    'emissions_indstr_high_t': emissions_indstr_high_t,
    'costs_without_revenues_eur': costs_without_revenues,
    'total_revenues_eur': total_revenues,
    'costs_net_after_revenues_eur': costs_net_after_revenues,
    'eua_costs_eur': total_emissions_tCO2 * df_eua[year],
    'CP_low_no_revs_eur_per_tCO2': CP_low_no_revs,
    'CP_high_no_revs_eur_per_tCO2': CP_high_no_revs,
    'CP_low_with_revs_eur_per_tCO2': CP_low_with_revs,
    'CP_high_with_revs_eur_per_tCO2': CP_high_with_revs
}

print(f"\nScenario: {row['scenario']} (year {row['year']})")
print(f" Demand (t/a): {row['demand_t']:,}")
print(f" Electricity need (MWh/a): {row['total_elec_MWh']:,.2f}" if row['total_elec_MWh'] is not None else " Electricity need: None")
print(f" Hub (MWh): {row['res_mwh']}, PPA (MWh): {row['ppa_mwh']:,.2f}, Grid (MWh): {row['grid_mwh']:,.2f}, Supply mismatch (MWh): {row['supply_mismatch_MWh']}")
print(f" Total emissions (tCO2): {row['total_emissions_tCO2']:,.2f}")
print(f" Emissions (tCO2/t MeOH): {row['emissions_tCO2_per_t']:,.2f}")
print(f" Costs w/o revs (€/a): {row['costs_without_revenues_eur']:,.2f}")
print(f" Total revenues (€/a): {row['total_revenues_eur']:,.2f}")
print(f" Costs net after revs (€/a): {row['costs_net_after_revenues_eur']:,.2f}")
print(f" EUA cost (€/a): {row['eua_costs_eur']:,.2f}")
print(" Necessary carbon prices (€/t CO2):")
print("  - vs industrial (low) WITHOUT revs:", row['CP_low_no_revs_eur_per_tCO2'])
print("  - vs industrial (high) WITHOUT revs:", row['CP_high_no_revs_eur_per_tCO2'])
print("  - vs industrial (low) WITH revs:", row['CP_low_with_revs_eur_per_tCO2'])
print("  - vs industrial (high) WITH revs:", row['CP_high_with_revs_eur_per_tCO2'])

370.0
0.8932
330.484

Scenario: Base (year 2019)
 Demand (t/a): 32,000
 Electricity need (MWh/a): 247,153.44
 Hub (MWh): 131805.9, PPA (MWh): 0.00, Grid (MWh): 115,347.54, Supply mismatch (MWh): 0
 Total emissions (tCO2): 42,909.29
 Emissions (tCO2/t MeOH): 1.34
 Costs w/o revs (€/a): 45,302,400.00
 Total revenues (€/a): 0.00
 Costs net after revs (€/a): 45,302,400.00
 EUA cost (€/a): 1,046,411.73
 Necessary carbon prices (€/t CO2):
  - vs industrial (low) WITHOUT revs: 1530.446017030231
  - vs industrial (high) WITHOUT revs: 712.9799888145387
  - vs industrial (low) WITH revs: 1530.446017030231
  - vs industrial (high) WITH revs: 712.9799888145387
